# Engine Caro 11×11 — sân chơi

Notebook để **xem và thử** engine bitboard (`caro/engine/`): vẽ bàn cờ, xem từng dãy bit, kiểm tra luật thắng,
ô thắng ngay, ứng viên, hàm đánh giá, Zobrist, đối xứng — và **tự chơi** bằng bàn cờ bấm được ở cuối.

**Mở notebook**

```bash
uv sync --group dev --extra analysis          # cài Jupyter (một lần)
uv run jupyter lab notebooks/engine_playground.ipynb
```

VS Code: mở file rồi chọn kernel `.venv` của dự án. Chạy lần lượt từ trên xuống (`Shift+Enter`).

Quy ước: ô (r, c) ↔ chỉ số `i = r·11 + c` (0…120); nước đi chính là chỉ số này. Hàng 0 ở trên, cột 0 bên trái.

In [1]:
import random
import time

from IPython.display import HTML, display

from caro.engine import CaroGame, IllegalMoveError, candidates, evaluate, has_five, iter_bits, winning_squares
from caro.engine.bitboard import CELLS, FULL, LINES, N, bit, dilate, index, popcount, rc
from caro.engine.symmetry import COUNT, canonical_key, transform_bb
from caro.engine.zobrist import full_key

## Công cụ vẽ

- `show(game_hoặc_(x, o), marks=…, title=…)` vẽ bàn cờ; `marks` là một bitboard các ô được tô vàng.
- `show_bits(b, title)` vẽ **một** bitboard dưới dạng lưới 0/1 — dùng để xem từng bước tính bit.
- `row(...)` đặt nhiều hình cạnh nhau.

In [2]:
CSS = '''
<style>
.cg{border-collapse:collapse;font:12px/1 monospace;margin:4px}
.cg td{width:22px;height:22px;text-align:center;border:1px solid #bbb;padding:0}
.cg th{font-weight:normal;color:#888;padding:0 3px}
.cg .x{color:#d62828;font-weight:bold}.cg .o{color:#1d4ed8;font-weight:bold}
.cg .mk{background:#fde68a}.cg .last{outline:2px solid #16a34a;outline-offset:-3px}
.cg .one{background:#334155;color:#fff}.cg .zero{color:#ccc}
.cg caption{caption-side:top;font:13px sans-serif;padding:2px;text-align:left}
.cgrow{display:flex;flex-wrap:wrap;gap:12px;align-items:flex-start}
</style>'''
display(HTML(CSS))


def _grid(cell, title=''):
    head = '<tr><th></th>' + ''.join(f'<th>{c}</th>' for c in range(N)) + '</tr>'
    body = ''.join('<tr><th>%d</th>%s</tr>' % (r, ''.join(cell(r, c) for c in range(N))) for r in range(N))
    return f'<table class="cg"><caption>{title}</caption>{head}{body}</table>'


def board_html(pos, marks=0, title=''):
    if isinstance(pos, CaroGame):
        (x, o), last = pos.bb, pos.last_move
    else:
        (x, o), last = pos, None

    def cell(r, c):
        i = index(r, c)
        cls = ['mk'] if (marks >> i) & 1 else []
        if i == last:
            cls.append('last')
        sym = 'X' if (x >> i) & 1 else 'O' if (o >> i) & 1 else ''
        if sym:
            cls.append(sym.lower())
        return f'<td class="{" ".join(cls)}">{sym}</td>'

    return _grid(cell, title)


def bits_html(b, title=''):
    return _grid(lambda r, c: '<td class="one">1</td>' if (b >> index(r, c)) & 1 else '<td class="zero">0</td>', title)


def show(pos, marks=0, title=''):
    display(HTML(board_html(pos, marks, title)))


def show_bits(b, title=''):
    display(HTML(bits_html(b, title)))


def row(*htmls):
    display(HTML('<div class="cgrow">' + ''.join(htmls) + '</div>'))


def cells(*rcs):
    m = 0
    for r, c in rcs:
        m |= bit(r, c)
    return m


def where(b):
    return [rc(i) for i in iter_bits(b)]

## 1. Một ván cờ

`step(i)` đi một nước và trả `(reward, done, info)` — reward theo góc nhìn người **vừa** đi.
`result()` luôn theo X: +1 X thắng, −1 O thắng, 0 hoà, `None` nếu chưa xong.

In [3]:
g = CaroGame()
for r, c in [(5, 5), (4, 4), (5, 6), (4, 5), (5, 7), (4, 6), (5, 8)]:   # X, O, X, O, …
    g.step(index(r, c))

show(g, title=repr(g))
print(g.render())

,0,1,2,3,4,5,6,7,8,9,10
0,,,,,,,,,,,
1,,,,,,,,,,,
2,,,,,,,,,,,
3,,,,,,,,,,,
4,,,,,O,O,O,,,,
5,,,,,,X,X,X,X,,
6,,,,,,,,,,,
7,,,,,,,,,,,
8,,,,,,,,,,,
9,,,,,,,,,,,


. . . . . . . . . . .
. . . . . . . . . . .
. . . . . . . . . . .
. . . . . . . . . . .
. . . . O O O . . . .
. . . . . X X X X . .
. . . . . . . . . . .
. . . . . . . . . . .
. . . . . . . . . . .
. . . . . . . . . . .
. . . . . . . . . . .


In [4]:
x, o = g.bb
ws = winning_squares(x, o)
print('X có ô thắng ngay:', where(ws), '→ tứ mở, O chỉ chặn được một đầu')
show(g, marks=ws, title='Ô vàng = ô X đi vào là thắng')

g.step(index(5, 4))                       # O chặn một đầu
print('X đi (5, 9):', g.step(index(5, 9)))
print('result =', g.result(), '| termination =', g.termination(), '| legal_actions =', g.legal_actions())
show(g, title=repr(g))

X có ô thắng ngay: [(5, 4), (5, 9)] → tứ mở, O chỉ chặn được một đầu


,0,1,2,3,4,5,6,7,8,9,10
0,,,,,,,,,,,
1,,,,,,,,,,,
2,,,,,,,,,,,
3,,,,,,,,,,,
4,,,,,O,O,O,,,,
5,,,,,,X,X,X,X,,
6,,,,,,,,,,,
7,,,,,,,,,,,
8,,,,,,,,,,,
9,,,,,,,,,,,


X đi (5, 9): (1.0, True, {'winner': 0})
result = 1 | termination = five | legal_actions = []


,0,1,2,3,4,5,6,7,8,9,10
0,,,,,,,,,,,
1,,,,,,,,,,,
2,,,,,,,,,,,
3,,,,,,,,,,,
4,,,,,O,O,O,,,,
5,,,,,O,X,X,X,X,X,
6,,,,,,,,,,,
7,,,,,,,,,,,
8,,,,,,,,,,,
9,,,,,,,,,,,


## 2. Bitboard thô

Mỗi bên là **một số nguyên 121 bit**. Bit `i` bật ↔ có quân ở ô `rc(i)`.

In [5]:
x, o = g.bb
print('X =', x)
print('X dạng nhị phân (bit 0 ở bên phải):', bin(x))
print('Ô của X:', where(x))
row(bits_html(x, 'bitboard X'), bits_html(o, 'bitboard O'), bits_html(~(x | o) & FULL, 'ô trống = ~(X|O) & FULL'))

X = 35740566642812256256
X dạng nhị phân (bit 0 ở bên phải): 0b11111000000000000000000000000000000000000000000000000000000000000
Ô của X: [(5, 5), (5, 6), (5, 7), (5, 8), (5, 9)]


,0,1,2,3,4,5,6,7,8,9,10
0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,1,1,1,1,1,0
6,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0


## 3. Kiểm tra thắng bằng 3 phép dịch

Với hướng có độ dịch `d` (ngang 1, dọc 11, ↘ 12, ↙ 10):

```
t = x & (x >> d)      # bit i: ô i và i+d có quân
t &= t >> 2d          # 4 ô liên tiếp
t &= x >> 4d          # 5 ô liên tiếp
t & START[d]          # bỏ kết quả lọt hàng
```

Bit `i` còn lại ↔ dãy 5 bắt đầu ở ô `i`. Ô (r, c) và ô (r, c+1) cách nhau 1 bit, nên `x >> 1` kéo ô bên phải về trùng ô bên trái.

In [6]:
def explain_five(x, d, name):
    start = dict(LINES)[d]
    t1 = x & (x >> d)
    t2 = t1 & (t1 >> (2 * d))
    t3 = t2 & (x >> (4 * d))
    row(bits_html(x, f'x ({name})'), bits_html(t1, f'x & x>>{d}'), bits_html(t2, f't & t>>{2 * d}'),
        bits_html(t3, f't & x>>{4 * d}'), bits_html(start, f'START[{d}]'), bits_html(t3 & start, 'kết quả'))
    print('has_five =', has_five(x), '| dãy bắt đầu ở', where(t3 & start))


explain_five(cells(*[(3, c) for c in range(2, 8)]), 1, 'dãy 6 ngang')

,0,1,2,3,4,5,6,7,8,9,10
0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0
3,0,0,1,1,1,1,1,1,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0


has_five = True | dãy bắt đầu ở [(3, 2), (3, 3)]


**Dãy giả do lọt hàng.** Các ô (2,8) (2,9) (2,10) (3,0) (3,1) có chỉ số 30…34 liên tiếp nhưng không thẳng hàng.
Công thức dịch vẫn ra một bit — mặt nạ START loại nó đi.

In [7]:
explain_five(cells((2, 8), (2, 9), (2, 10), (3, 0), (3, 1)), 1, 'giả, lọt hàng')

,0,1,2,3,4,5,6,7,8,9,10
0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,1,1
3,1,1,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0


has_five = False | dãy bắt đầu ở []


In [8]:
explain_five(cells(*[(1 + k, 2 + k) for k in range(5)]), 12, 'chéo ↘')
explain_five(cells(*[(2 + k, 8 - k) for k in range(5)]), 10, 'chéo ↙')

,0,1,2,3,4,5,6,7,8,9,10
0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,0,0,0,0,0,0,0
2,0,0,0,1,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0
5,0,0,0,0,0,0,1,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0


has_five = True | dãy bắt đầu ở [(1, 2)]


,0,1,2,3,4,5,6,7,8,9,10
0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,0,0
3,0,0,0,0,0,0,0,1,0,0,0
4,0,0,0,0,0,0,1,0,0,0,0
5,0,0,0,0,0,1,0,0,0,0,0
6,0,0,0,0,1,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,0


has_five = True | dãy bắt đầu ở [(2, 8)]


## 4. Ô thắng ngay — `winning_squares(me, opp)`

Ô trống mà `me` đặt vào là có ≥ 5 liền. Đếm số ô này là đủ để nhận ra các thế cờ:
1 ô = tứ (bắt buộc đối thủ chặn), ≥ 2 ô = tứ mở / tứ kép (không chặn nổi).

In [9]:
patterns = {
    'tứ mở  _XXXX_': ([(5, c) for c in range(3, 7)], []),
    'tứ bị chặn OXXXX_': ([(5, c) for c in range(3, 7)], [(5, 2)]),
    'tứ có lỗ XX_XX': ([(5, 2), (5, 3), (5, 5), (5, 6)], []),
    'tứ kép (2 hướng)': ([(5, 3), (5, 4), (5, 5), (5, 6), (6, 2), (7, 2), (8, 2), (9, 2)], [(5, 2), (5, 7), (10, 2)]),
}
htmls = []
for name, (xs, os_) in patterns.items():
    x, o = cells(*xs), cells(*os_)
    ws = winning_squares(x, o)
    htmls.append(board_html((x, o), ws, f'{name}: {popcount(ws)} ô'))
row(*htmls)

,0,1,2,3,4,5,6,7,8,9,10
0,,,,,,,,,,,
1,,,,,,,,,,,
2,,,,,,,,,,,
3,,,,,,,,,,,
4,,,,,,,,,,,
5,,,,X,X,X,X,,,,
6,,,,,,,,,,,
7,,,,,,,,,,,
8,,,,,,,,,,,
9,,,,,,,,,,,


## 5. Ứng viên — `candidates(me, opp, radius)`

Chỉ xét ô trống trong bán kính `radius` quanh quân đã có (phép giãn nở `dilate` lặp `radius` lần).
Quân sát mép phải không làm ứng viên "lọt" sang mép trái.

In [10]:
x, o = cells((5, 5), (3, 10)), cells((5, 6))
row(board_html((x, o), candidates(x, o, 1), f'radius 1: {popcount(candidates(x, o, 1))} ô'),
    board_html((x, o), candidates(x, o, 2), f'radius 2: {popcount(candidates(x, o, 2))} ô'),
    bits_html(dilate(bit(3, 10)), 'dilate một ô ở mép phải'))
print('Bàn trống → ứng viên duy nhất là ô giữa:', where(candidates(0, 0)))

,0,1,2,3,4,5,6,7,8,9,10
0,,,,,,,,,,,
1,,,,,,,,,,,
2,,,,,,,,,,,
3,,,,,,,,,,,X
4,,,,,,,,,,,
5,,,,,,X,O,,,,
6,,,,,,,,,,,
7,,,,,,,,,,,
8,,,,,,,,,,,
9,,,,,,,,,,,


Bàn trống → ứng viên duy nhất là ô giữa: [(5, 5)]


## 6. Hàm đánh giá — `evaluate(me, opp)`

Cộng điểm cho mọi cửa sổ 5 ô **chưa bị đối thủ chạm vào**, theo số quân của mình trong cửa sổ:
trọng số `[0, 1, 8, 64, 512]`. Ba mở có nhiều cửa sổ sống hơn ba bị chặn nên điểm cao hơn — không cần luật riêng.

In [11]:
cases = {
    'ba mở': (cells((5, 4), (5, 5), (5, 6)), 0),
    'ba bị chặn 1 đầu': (cells((5, 4), (5, 5), (5, 6)), cells((5, 3))),
    'ba bị chặn 2 đầu': (cells((5, 4), (5, 5), (5, 6)), cells((5, 3), (5, 7))),
    'ba sát mép': (cells((5, 0), (5, 1), (5, 2)), 0),
}
row(*[board_html(p, title=f'{k}: {evaluate(*p)}') for k, p in cases.items()])

,0,1,2,3,4,5,6,7,8,9,10
0,,,,,,,,,,,
1,,,,,,,,,,,
2,,,,,,,,,,,
3,,,,,,,,,,,
4,,,,,,,,,,,
5,,,,,X,X,X,,,,
6,,,,,,,,,,,
7,,,,,,,,,,,
8,,,,,,,,,,,
9,,,,,,,,,,,


## 7. Lùi nước, sao chép, Zobrist

Khoá Zobrist cập nhật bằng XOR mỗi nước; lùi hết về bàn trống thì khoá về 0.
Hai thứ tự đi khác nhau dẫn tới cùng một thế → cùng khoá (dùng cho bảng chuyển vị).

In [12]:
rng = random.Random(1)
g = CaroGame()
for _ in range(12):
    g.step(rng.choice(g.legal_actions()))
print('khoá sau 12 nước :', hex(g.key), '| tính lại từ đầu:', hex(full_key(*g.bb, g.side)))

c = g.clone()
c.step(c.legal_actions()[0])
print('clone đi thêm 1 nước, bản gốc vẫn', len(g.history), 'nước')

while g.history:
    g.undo()
print('lùi hết →', g, '| key =', g.key)

a = CaroGame.from_moves([60, 61, 70, 71])
b = CaroGame.from_moves([70, 71, 60, 61])
print('khác thứ tự, cùng thế → cùng khoá:', a.key == b.key)

khoá sau 12 nước : 0x75c31ae037c8f02e | tính lại từ đầu: 0x75c31ae037c8f02e
clone đi thêm 1 nước, bản gốc vẫn 12 nước
lùi hết → CaroGame(0 nước, X đi) | key = 0
khác thứ tự, cùng thế → cùng khoá: True


## 8. Đối xứng D4

8 phép (xoay 0/90/180/270°, lật ngang/dọc, lật 2 đường chéo). Dùng để nhân dữ liệu huấn luyện ×8 và
tính khoá chuẩn hoá — 8 thế đối xứng cho cùng một `canonical_key`.

In [13]:
x, o = cells((2, 2), (2, 3), (3, 3), (2, 5)), cells((4, 4), (1, 2))
names = ['giữ nguyên', 'xoay 90°', 'xoay 180°', 'xoay 270°', 'lật trái–phải', 'lật trên–dưới', 'chéo chính', 'chéo phụ']
row(*[board_html((transform_bb(x, k), transform_bb(o, k)), title=names[k]) for k in range(COUNT)])
keys = {canonical_key(transform_bb(x, k), transform_bb(o, k), 0) for k in range(COUNT)}
print('Số khoá chuẩn hoá khác nhau qua 8 phép:', len(keys))

,0,1,2,3,4,5,6,7,8,9,10
0,,,,,,,,,,,
1,,,O,,,,,,,,
2,,,X,X,,X,,,,,
3,,,,X,,,,,,,
4,,,,,O,,,,,,
5,,,,,,,,,,,
6,,,,,,,,,,,
7,,,,,,,,,,,
8,,,,,,,,,,,
9,,,,,,,,,,,


Số khoá chuẩn hoá khác nhau qua 8 phép: 1


## 9. Nước bất hợp lệ

Mọi nước sai báo `IllegalMoveError` và **không** làm đổi trạng thái ván.

In [14]:
g = CaroGame.from_moves([60])
for bad in [60, -1, 121, 'a', 2.5]:
    try:
        g.step(bad)
    except IllegalMoveError as e:
        print(f'{bad!r:>5} → IllegalMoveError: {e}')
print(g)

   60 → IllegalMoveError: nước 60 không hợp lệ
   -1 → IllegalMoveError: nước -1 không hợp lệ
  121 → IllegalMoveError: nước 121 không hợp lệ
  'a' → IllegalMoveError: nước 'a' không phải số nguyên
  2.5 → IllegalMoveError: nước 2.5 không phải số nguyên
CaroGame(1 nước, O đi)


## 10. Mô phỏng ván ngẫu nhiên

Hai bên đi ngẫu nhiên trong các ô trống: đo tốc độ engine và xem lợi thế đi trước ở mức "không biết chơi".
Đổi `GAMES` hoặc chọn nước trong `candidates` thay vì mọi ô trống để thấy khác biệt.

In [15]:
GAMES = 1000
rng = random.Random(0)
stats = {1: 0, -1: 0, 0: 0}
moves = 0
t0 = time.perf_counter()
for _ in range(GAMES):
    g = CaroGame()
    while not g.is_terminal():
        g.step(rng.choice(g.legal_actions()))
    stats[g.result()] += 1
    moves += len(g.history)
dt = time.perf_counter() - t0
print(f'{GAMES} ván trong {dt:.2f} s ({GAMES / dt:.0f} ván/s, {moves / dt / 1000:.0f}k nước/s)')
print(f'X thắng {stats[1] / GAMES:.1%} · O thắng {stats[-1] / GAMES:.1%} · hoà {stats[0] / GAMES:.1%} · '
      f'trung bình {moves / GAMES:.1f} nước/ván')

1000 ván trong 0.78 s (1282 ván/s, 90k nước/s)
X thắng 52.7% · O thắng 47.3% · hoà 0.0% · trung bình 69.8 nước/ván


## 11. Tự chơi

Bấm vào ô để đi. Tuỳ chọn:
- **Máy cầm O**: sau mỗi nước của bạn, máy đi ngẫu nhiên trong vùng ứng viên — nhưng luôn thắng ngay nếu được và chặn nếu bị đe doạ.
- **Gợi ý**: tô vàng ô thắng ngay của bên đang đi (đi vào là thắng) và tô đỏ nhạt ô phải chặn.

Cần `ipywidgets` (có sẵn khi cài `--extra analysis`). Nếu không hiện bàn bấm được, dùng các hàm `play(r, c)` / `back()` ở ô kế tiếp.

In [16]:
import ipywidgets as W

game = CaroGame()
rng_ai = random.Random(42)


def simple_ai(g):
    me, opp = g.me_opp()
    for mask in (winning_squares(me, opp), winning_squares(opp, me), candidates(me, opp, 1)):
        if mask:
            return rng_ai.choice(list(iter_bits(mask)))
    return rng_ai.choice(g.legal_actions())


buttons = [W.Button(layout=W.Layout(width='34px', height='34px', padding='0')) for _ in range(CELLS)]
status = W.HTML()
ai_o = W.Checkbox(value=True, description='Máy cầm O', indent=False)
hints = W.Checkbox(value=True, description='Gợi ý', indent=False)
btn_new, btn_undo = W.Button(description='Ván mới'), W.Button(description='Lùi')


def redraw():
    me, opp = game.me_opp()
    win = winning_squares(me, opp) if hints.value and not game.is_terminal() else 0
    must = winning_squares(opp, me) if hints.value and not game.is_terminal() else 0
    x, o = game.bb
    for i, b in enumerate(buttons):
        b.description = 'X' if (x >> i) & 1 else 'O' if (o >> i) & 1 else ''
        b.style.font_weight = 'bold'
        b.style.text_color = '#d62828' if (x >> i) & 1 else '#1d4ed8'
        b.style.button_color = ('#86efac' if i == game.last_move else '#fde68a' if (win >> i) & 1
                                else '#fecaca' if (must >> i) & 1 else None)
        b.disabled = bool(((x | o) >> i) & 1) or game.is_terminal()
    if game.is_terminal():
        msg = {1: 'X thắng 🎉', -1: 'O thắng', 0: 'Hoà'}[game.result()]
    else:
        msg = f'{"XO"[game.side]} đến lượt'
    status.value = f'<b>{msg}</b> · {len(game.history)} nước · đánh giá cho X: {evaluate(game.bb[0], game.bb[1])}'


def on_click(i):
    def handler(_):
        if game.is_terminal():
            return
        game.step(i)
        if ai_o.value and not game.is_terminal() and game.side == 1:
            game.step(simple_ai(game))
        redraw()
    return handler


def on_new(_):
    game.reset()
    redraw()


def on_undo(_):
    n = 2 if ai_o.value and len(game.history) >= 2 and game.side == 0 else 1
    for _ in range(min(n, len(game.history))):
        game.undo()
    redraw()


for i, b in enumerate(buttons):
    b.on_click(on_click(i))
btn_new.on_click(on_new)
btn_undo.on_click(on_undo)
hints.observe(lambda _: redraw(), 'value')

redraw()
grid = W.GridBox(buttons, layout=W.Layout(grid_template_columns=f'repeat({N}, 34px)', grid_gap='2px'))
display(W.VBox([W.HBox([btn_new, btn_undo, ai_o, hints]), status, grid]))

In [17]:
# Chơi bằng lệnh (không cần widget): dùng chung biến `game` với bàn bấm ở trên
def play(r, c):
    game.step(index(r, c))
    if ai_o.value and not game.is_terminal() and game.side == 1:
        game.step(simple_ai(game))
    redraw()
    me, opp = game.me_opp()
    show(game, marks=winning_squares(me, opp) | winning_squares(opp, me), title=repr(game))


def back():
    on_undo(None)
    show(game, title=repr(game))


# ví dụ: play(5, 5)